# Formativa 1 — Framingham Heart Study

Grupo 3 — MCDI501

Análisis exploratorio de datos e inferencia sobre el dataset Framingham Heart Study (predicción de enfermedad coronaria a 10 años, `TenYearCHD`).

In [ ]:
import sys
print(sys.executable)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import kagglehub, shutil
from pathlib import Path

path = kagglehub.dataset_download("dileep070/heart-disease-prediction-using-logistic-regression")
destino = Path("../../data/raw")
destino.mkdir(parents=True, exist_ok=True)
shutil.copy(Path(path) / "framingham.csv", destino / "framingham.csv")
print("Copiado en:", (destino / "framingham.csv").resolve())

In [ ]:
df = pd.read_csv("../../data/raw/framingham.csv")
print(df.shape)
df.info()
df.head()

### Semilla, variable objetivo y estilo de las figuras
Semilla fija `42`. Paleta común para todas las figuras del informe (colores UNAB).

In [ ]:
from scipy import stats

SEED = 42
np.random.seed(SEED)
TARGET = "TenYearCHD"

# Estilo común de las figuras (paleta UNAB)
NAVY, RED, GREY, BG = "#0d2741", "#ae0000", "#8c8578", "#f6f1ea"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200,
    "font.size": 10, "axes.facecolor": BG,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.grid": True, "grid.color": "#ddd5c6", "axes.axisbelow": True,
})
print("Semilla:", SEED, "| Objetivo:", TARGET)

## 1. Estadística descriptiva (ID1.2)

### 1.1 Tipos de variables y medidas resumen
Una variable objetivo binaria (`TenYearCHD`), cinco indicadores binarios, una variable ordinal, un conteo
y siete variables numéricas continuas o cuasi-continuas.

In [ ]:
tipos = pd.Series({
    "TenYearCHD": "Binaria (objetivo)",
    "male": "Binaria", "currentSmoker": "Binaria", "BPMeds": "Binaria",
    "prevalentStroke": "Binaria", "prevalentHyp": "Binaria", "diabetes": "Binaria",
    "education": "Ordinal (1-4)",
    "cigsPerDay": "Conteo",
    "age": "Cuasi-continua",
    "totChol": "Continua", "sysBP": "Continua", "diaBP": "Continua",
    "BMI": "Continua", "heartRate": "Continua", "glucose": "Continua",
}, name="Tipo")
tipos.to_frame()

In [ ]:
# Tabla 1 del informe: media, mediana, desviación estándar, mínimo y máximo
cuant = ["age", "cigsPerDay", "totChol", "sysBP", "diaBP", "BMI", "heartRate", "glucose"]
desc = df[cuant].describe().T[["mean", "50%", "std", "min", "max"]]
desc.columns = ["Media", "Mediana", "Desv. est.", "Mínimo", "Máximo"]
desc.round(2)

### 1.2 Calidad de datos
Desbalance del objetivo, datos faltantes, duplicados y valores extremos.

In [ ]:
n = len(df)
print(f"Prevalencia de CHD a 10 años : {df[TARGET].mean()*100:.2f}%  ({df[TARGET].sum():,} de {n:,})")
print(f"Filas duplicadas             : {df.duplicated().sum():,}")
print()
falt = pd.DataFrame({"n faltantes": df.isna().sum(), "% faltante": (df.isna().mean() * 100).round(2)})
print(falt[falt["n faltantes"] > 0].sort_values("n faltantes", ascending=False))
print(f"\nFilas con al menos un faltante: {df.isna().any(axis=1).sum():,} ({df.isna().any(axis=1).mean()*100:.1f}%)")

# Valores extremos (regla de Tukey: fuera de Q1-1.5*IQR, Q3+1.5*IQR)
print("\nValores extremos según Tukey:")
for c in ["totChol", "sysBP", "diaBP", "BMI", "glucose"]:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    k = ((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum()
    print(f"  {c:10s}: {k:4d} ({k/df[c].notna().sum()*100:.1f}%)")

### Figura 1 - Variable objetivo y datos faltantes

In [ ]:
from pathlib import Path
Path("figuras").mkdir(exist_ok=True)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# (a) distribución de la variable objetivo
vc = df[TARGET].value_counts().sort_index()
labels = ["Sin CHD (0)", "Con CHD (1)"]
ax[0].barh(labels, vc.values, color=[NAVY, RED], height=0.5)
ax[0].invert_yaxis()
for i, v in enumerate(vc.values):
    ax[0].text(v + 50, i, f"{v:,}  ({v/n*100:.1f}%)", va="center", fontsize=9)
ax[0].set_xlim(0, vc.max() * 1.25)
ax[0].set_xlabel("Frecuencia")
ax[0].set_title("(a) Distribución de la variable objetivo")
ax[0].grid(axis="y", visible=False)

# (b) porcentaje de faltantes por variable
miss = (df.isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]
y = np.arange(len(miss))
ax[1].hlines(y, 0, miss.values, color="#bdb3a3", lw=2.5)
ax[1].scatter(miss.values, y, s=90, color=RED, edgecolor="#333333", linewidth=0.6, zorder=3)
for i, v in enumerate(miss.values):
    ax[1].text(v + 0.2, i, f"{v:.1f}%", va="center", fontsize=9)
ax[1].set_yticks(y)
ax[1].set_yticklabels(miss.index, fontsize=9)
ax[1].invert_yaxis()
ax[1].set_xlim(0, miss.max() * 1.25)
ax[1].set_xlabel("% faltante")
ax[1].set_title("(b) Datos faltantes por variable")
ax[1].grid(axis="y", visible=False)

plt.tight_layout()
plt.savefig("figuras/fig_panorama.png", bbox_inches="tight")
plt.show()

### Figura 2 - Distribución de tres variables representativas
El colesterol total se recorta al percentil 99 solo para visualizarlo; los cálculos usan todos los datos.

In [ ]:
chol = df["totChol"].dropna()
sbp = df["sysBP"]
paneles = [
    (df["age"], "Edad", "años", "media", df["age"].mean()),
    (chol.clip(upper=chol.quantile(0.99)), "Colesterol total (≤P99)", "mg/dL", "mediana", chol.median()),
    (sbp, "Presión sistólica", "mmHg", "mediana", sbp.median()),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (x, titulo, xlab, nombre, ref) in zip(axes, paneles):
    ax.hist(x, bins=40, histtype="stepfilled", color=NAVY, alpha=0.3, edgecolor=NAVY, linewidth=1.2)
    ax.axvline(ref, color=RED, ls="--", lw=1.8)
    ax.text(ref, ax.get_ylim()[1] * 0.93, f" {nombre}={ref:,.0f}", color=RED, fontsize=9, va="center")
    ax.set_title(titulo)
    ax.set_xlabel(xlab)
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("Frecuencia")

plt.tight_layout()
plt.savefig("figuras/fig_distribuciones.png", bbox_inches="tight")
plt.show()

### Figura 3 - Matriz de correlaciones (triángulo inferior)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

corr = df.drop(columns=TARGET).corr()
k = len(corr)
mask = np.triu(np.ones((k, k), dtype=bool), k=1)   # oculta el triángulo superior
cmap = LinearSegmentedColormap.from_list("div", [RED, "#ede4d6", NAVY])

fig, ax = plt.subplots(figsize=(8, 7))
ax.set_facecolor("white")
ax.grid(False)
im = ax.pcolormesh(np.ma.masked_array(corr.values, mask), cmap=cmap, vmin=-1, vmax=1,
                   edgecolors="white", linewidth=1.5)
ax.invert_yaxis()
for i in range(k):
    for j in range(i + 1):
        v = corr.values[i, j]
        ax.text(j + 0.5, i + 0.5, f"{v:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(v) > 0.5 else "#333333")
ax.set_xticks(np.arange(k) + 0.5)
ax.set_xticklabels(corr.columns, rotation=90, fontsize=7.5)
ax.set_yticks(np.arange(k) + 0.5)
ax.set_yticklabels(corr.columns, fontsize=7.5)
ax.set_title("Matriz de correlaciones entre variables")
cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label("Correlación de Pearson")

plt.tight_layout()
plt.savefig("figuras/fig_correlacion.png", bbox_inches="tight")
plt.show()

# Pares con mayor |r| fuera de la diagonal
pares = corr.where(np.tril(np.ones((k, k), dtype=bool), k=-1)).stack()
pares.reindex(pares.abs().sort_values(ascending=False).index).head(5).round(3)

In [ ]:
df.corr()[TARGET].drop(TARGET).sort_values().round(4)